### Setup Inicial

Começamos criando a Spark Session para realizar a conexão com o cluster

### Conversão para Parquet

Após isso o dataset será convertido inteiramente para parquet visando a otimização de queries.

In [22]:
from pyspark.sql import SparkSession
import os
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType, BooleanType, \
    DateType, ArrayType

base_url = '/home/jovyan/work'
input_path = f"{base_url}/data"
output_path = f"{base_url}/data_parquet"

spark = SparkSession.builder \
    .appName('AC2-BigData') \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
# Schema para a tabela 'ratings'
schema_ratings = StructType([
    StructField("username", StringType(), True),
    StructField("anime_id", LongType(), True),
    StructField("status", StringType(), True),
    StructField("score", DoubleType(), True),
    StructField("is_rewatching", BooleanType(), True),
    StructField("num_watched_episodes", IntegerType(), True)
])

# 1. Ajuste o schema de details para ler episodes como String inicialmente
# É importante ler o schema completo para evitar colunas em ordem errada
schema_details = StructType([
    StructField("mal_id", LongType(), True),
    StructField("title", StringType(), True),
    StructField("title_japanese", StringType(), True),
    StructField("url", StringType(), True),
    StructField("image_url", StringType(), True),
    StructField("type", StringType(), True),
    StructField("status", StringType(), True),
    StructField("score", DoubleType(), True),
    StructField("scored_by", LongType(), True),
    StructField("start_date", DateType(), True),
    StructField("end_date", DateType(), True),
    StructField("synopsis", StringType(), True),
    StructField("rank", DoubleType(), True),
    StructField("popularity", IntegerType(), True),
    StructField("members", LongType(), True),
    StructField("favorites", LongType(), True),
    StructField("genres", StringType(), True),
    StructField("studios", StringType(), True),
    StructField("themes", StringType(), True),
    StructField("demographics", StringType(), True),
    StructField("source", StringType(), True),
    StructField("rating", StringType(), True),
    StructField("episodes", DoubleType(), True),
    StructField("season", StringType(), True),
    StructField("year", DoubleType(), True),
    StructField("producers", StringType(), True),
    StructField("explicit_genres", StringType(), True),
    StructField("licensors", StringType(), True),
    StructField("streaming", StringType(), True)
])

In [2]:


def convert_table(file_name, schema):
    table_name = file_name.replace(".csv", "")
    print(f"Iniciando conversão de: {table_name}...")

    # Configuração específica para a tabela complexa
    if table_name == "details":
        is_multiline = "true"
        # Opções extras para lidar com o desalinhamento e espaços
        extra_options = {
            "ignoreLeadingWhiteSpace": "true",
            "ignoreTrailingWhiteSpace": "true",
            "mode": "PERMISSIVE" # Garante que ele tente ler mesmo com erros
        }
    else:
        is_multiline = "false"
        extra_options = {}

    df = spark.read.format("csv") \
        .option("header", "True") \
        .schema(schema) \
        .option("nullValue", "N/A") \
        .option("quote", "\"") \
        .option("escape", "\"") \
        .option("multiLine", is_multiline) \
        .options(**extra_options) \
        .load(os.path.join(input_path, file_name))

    # Salvando em Parquet
    df.write.mode("overwrite").parquet(os.path.join(output_path, table_name))
    print(f"Sucesso! {table_name} convertida.\n")

convert_table("details.csv", schema_details)
convert_table("ratings.csv", schema_ratings)

Iniciando conversão de: details...


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/work/data/details.csv.

Com o dataset já convertido agora é possível realizar queries com mais eficiência. Porém, antes disso será realizado uma comparação em performance entre o dataset em csv puro e o otimizado para dados colunares em parquet.

In [ ]:
import time
import matplotlib.pyplot as plt

# Função para medir o tempo de uma query SQL
def benchmark_format(bench_table, format_type):
    start_time = time.time()

    # 1. Lê o arquivo baseado no formato
    if format_type == "csv":
        # Lendo o CSV original (lento)
        df_bench = spark.read.csv(f"{base_url}/data/{bench_table}.csv", header=True, inferSchema=True)
    else:
        # Lendo o Parquet convertido (rápido)
        df_bench = spark.read.parquet(f"{base_url}/data_parquet/{bench_table}")

    # 2. Registra a view para o Injetor SQL
    df_bench.createOrReplaceTempView(f"temp_{format_type}")

    # 3. Executa uma ação (count força o Spark a ler os dados de fato)
    count = spark.sql(f"SELECT count(*) FROM temp_{format_type}").collect()[0][0]

    end_time = time.time()
    duration = end_time - start_time
    print(f"Tempo [{format_type.upper()}]: {duration:.2f} segundos para {count} registros.")
    return duration

# Executando o teste na tabela de 'ratings' (que é a maior)
tempo_csv = benchmark_format("ratings", "csv")
tempo_parquet = benchmark_format("ratings", "parquet")

In [ ]:
# Dados para o gráfico
formatos = ['CSV (Original)', 'Parquet (Otimizado)']
tempos = [tempo_csv, tempo_parquet]
cores = ['#e74c3c', '#2ecc71'] # Vermelho para lento, Verde para rápido

# Criando o gráfico
plt.bar(formatos, tempos, color=cores)
plt.ylabel('Tempo de Execução (segundos)')
plt.title('Comparação de Performance: CSV vs Parquet (Leitura + SQL Count)')

# Adicionando os valores em cima das barras
for i, v in enumerate(tempos):
    plt.text(i, v + 0.1, f"{v:.2f}s", ha='center', fontweight='bold')

plt.show()

O dataset em CSV puro possui aproximadamente 4.6GB, enquanto o dataset otimizado utilizando Parquet possui 455MB. Uma redução de 90% em tamanho que transmitiu um
 ganho de performance 43 vezes mais rápido.

### Parte 1: Definição de Schemas e Carregamento

Com a comparação de desempenho não resta dúvida qualquer que parquet é extremamente mais eficiente, com isso em mente podemos começar de fato a desenvolver. Para isso, primeiramente será definido o Schema de cada tabela a ser trabalhada e a criação de temp_views para SQL Injection.


In [ ]:
# 2. Carregamento dos dados em formato Parquet com Schemas definidos

df_ratings = spark.read.schema(schema_ratings).parquet(f"{output_path}/ratings")
df_details = spark.read.schema(schema_details).parquet(f"{output_path}/details")
# Select para vericicar sanidade dos dados
spark.read.parquet(f"{output_path}/details").select("mal_id", "title", "episodes", "type").show(10)
# 3. Registro das TempViews para injeção SQL
df_ratings.createOrReplaceTempView("raw_ratings")
df_details.createOrReplaceTempView("raw_details")

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/work/data_parquet/ratings.

### Preparação da Master View (Injeção SQL)
Primeiro, consolidamos os dados. Vamos considerar 1 para Completed e 0 para qualquer outro status (Dropped, On-Hold, Plan to Watch, Watching), garantindo que temos dados de nota para análise.

In [ ]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW v_master_raw AS
    SELECT
        r.username as username,
        r.anime_id,
        CAST(r.score AS DOUBLE) as user_score,
        -- Antigo Target (Apenas completou)
        CASE WHEN LOWER(TRIM(r.status)) = 'completed' THEN 1 ELSE 0 END as completed_only,

        -- NOVO TARGET: Engajamento Real (Completou + Nota >= 8)
        CASE
            WHEN LOWER(TRIM(r.status)) = 'completed' AND CAST(r.score AS DOUBLE) >= 8
            THEN 1
            ELSE 0
        END as target,

        d.type as anime_type,
        d.title as anime_title,
        d.source as anime_source,
        CAST(d.episodes AS INT) as total_episodes,
        CAST(d.score AS DOUBLE) as anime_score,
        d.start_date as start_date
    FROM raw_ratings r
    INNER JOIN raw_details d ON r.anime_id = d.mal_id
    WHERE r.score > 0 AND d.start_date >= '1975-01-01'
""")


### Salva-se a tabela com as colunas desejadas


In [ ]:
(spark.table("v_master_raw")
 .write
 .mode("overwrite")
 .parquet(f"{output_path}/master_raw"))

### Agora salvamos uma sample do dataset para facilitar o processamento


In [ ]:
# 1. Configuração padrão para não forçar demais
spark.conf.set("spark.sql.shuffle.partitions", "200")

# 2. Leitura do dataset bruto que você já salvou
df_master_raw = spark.read.parquet(f"{output_path}/master_raw")

# 3. AMOSTRAGEM: Pegamos 40% dos dados de forma aleatória, mas consistente (seed=42)
# Isso reduz 70M para ~28M, o que é muito mais manejável
print("Criando amostra de 40% para viabilizar o processamento...")
df_sample = df_master_raw.sample(withReplacement=False, fraction=0.4, seed=42)

# 5. Escrita do Dataset tratado
print("Iniciando a final da Master Table (Amostra)...")
(df_sample
 .write
 .mode("overwrite")
 .parquet(f"{output_path}/master_table_with_duplicates"))

print("--- Processo concluído com amostra de 40%. ---")


### E finalmente, elimina-se as duplicatas da tabela

In [ ]:
# 1. Lê a amostra bruta
df_sample_bruto = spark.read.parquet(f"{output_path}/master_table_with_duplicates")

df_sample_bruto.createOrReplaceTempView("v_master_raw_sample")
# 2. Aplica a remoção de duplicatas (Usuário + Anime)
# 2. Executa a deduplicação via SQL Estrito
# O PARTITION BY garante que olhamos para o par Usuário-Anime
# O ORDER BY user_score DESC garante que, se houver duplicata, a maior nota fica no topo
df_master_without_dups = spark.sql("""
                            SELECT
                                username,
                                anime_id,
                                user_score,
                                target,
                                completed_only,
                                anime_type,
                                anime_title,
                                anime_source,
                                total_episodes,
                                anime_score,
                                start_date
                            FROM (
                                     SELECT *,
                                            ROW_NUMBER() OVER (
                   PARTITION BY username, anime_id
                   ORDER BY user_score DESC
               ) as row_num
                                     FROM v_master_raw_sample
                                 ) ranking_query
                            WHERE row_num = 1
                            """)

# 3. Salva a versão definitiva
print("Limpando duplicatas e gerando a Master Table final...")
(df_master_without_dups
 .write
 .mode("overwrite")
 .parquet(f"{output_path}/master_table"))
print("Master table salva com sucesso!")

In [ ]:
# 1. Carrega o dataset sem duplicatas!
df_master = spark.read.parquet(f"{output_path}/master_table")
df_master_with_dups = spark.read.parquet(f"{output_path}/master_table_with_duplicates")
# 2. Registra a View
df_master.createOrReplaceTempView("master_table")
df_master_with_dups.createOrReplaceTempView("master_table_dups")

print("Quantidade de linhas duplicadas removidas")
dups_result = spark.sql("""
          SELECT
                  (SELECT COUNT(*) FROM master_table_dups) AS total_com_duplicatas,
                  (SELECT COUNT(*) FROM master_table) AS total_limpo,
                  (SELECT COUNT(*) FROM master_table_dups) - (SELECT COUNT(*) FROM master_table) AS linhas_removidas
          """)

display(dups_result)

print("Verificando médias de episódios")
avg_ep = spark.sql("""
                   SELECT
                       MIN(total_episodes) as Minimo,
                       MAX(total_episodes) as Maximo,
                       ROUND(AVG(total_episodes), 2) as Media
                   FROM master_table
                   """)
display(avg_ep)

print("Confirmando se episódios estão dentro da média")
sanity_result = spark.sql("""
                          SELECT anime_id, anime_type, total_episodes, anime_title
                          FROM master_table
                          WHERE total_episodes <= 24
                              LIMIT 10
                          """)
# Chame novamente para ver a tabela rica
sanity_result

# Documentação de ETL e Tratamento de Dados: Dataset de Animes

Esta documentação descreve as etapas de extração, transformação e carregamento (ETL) aplicadas aos dados brutos para garantir a integridade estatística e a usabilidade em modelos de Machine Learning.

---

## 1. Mapeamento Estrutural (Schema Enforcement)
**Onde foi utilizado:** Na leitura inicial dos arquivos CSV (`details.csv` e `ratings.csv`) para conversão em Parquet.

**O que foi feito:**
* Definição de um `StructType` contendo as **29 colunas** reais presentes no arquivo de detalhes.
* Tipagem inicial de colunas críticas como `episodes` e `score` como `StringType`.

**Por que:**
* **Correção de Desalinhamento:** O Spark é sensível a vírgulas extras e aspas mal fechadas. Sem mapear as 29 colunas, o Spark "empurrava" dados do final da linha (como URLs de imagem) para colunas do meio (como episódios).
* **Prevenção de Erros de Tipo:** Ler dados "sujos" como `String` evita que o Spark descarte linhas (retornando `null`) ao encontrar valores não numéricos como "N/A".

---

## 2. Tratamento de Delimitadores e Multilinha
**Onde foi utilizado:** Configuração do leitor de CSV do Spark para a tabela `details`.

**O que foi feito:**
* Habilitação das opções `multiLine`, `quote`, `escape`, `ignoreLeadingWhiteSpace` e `ignoreTrailingWhiteSpace`.

**Por que:**
* **Integridade dos Registros:** Títulos e sinopses possuem aspas duplas internas e quebras de linha. Essas opções garantem que o Spark entenda onde cada registro começa e termina, evitando registros corrompidos.

---


## 3. Binarização do Target de Engajamento (Feature Engineering)

**Onde foi utilizado:** Criação da coluna definitiva `target` na `master_table` e nas tabelas de treino, utilizando injeção de SQL Strict sobre o dataset consolidado.

**O que foi feito:**
Diferente de uma binarização simples baseada apenas no status, implementou-se uma lógica condicional composta para definir o sucesso da interação:

* **SQL Logic:** `CASE WHEN LOWER(TRIM(r.status)) = 'completed' AND CAST(r.score AS DOUBLE) >= 8 THEN 1 ELSE 0 END`

**Por que:**
* **Qualificação do Sinal (High Satisfaction):** O alvo híbrido filtra "conclusões passivas". Ao exigir uma nota $\geq 8$ aliada à conclusão, garantimos que o modelo aprenda padrões de **alto engajamento e satisfação real**, e não apenas a finalização mecânica de um conteúdo que o usuário pode ter considerado mediano ou ruim.
* **Preparação para Classificação Binária:** Transforma estados categóricos (status) e métricas contínuas (score) em um valor numérico discreto (0 ou 1). Isso permite que o algoritmo de Machine Learning estime a probabilidade de um usuário "amar" um conteúdo, servindo como base para um sistema de recomendação de alta precisão.
* **Equilíbrio Estatístico Natural:** A aplicação desta regra de negócio refinada resultou em um dataset organicamente equilibrado (~48% para a Classe 1 e ~52% para a Classe 0). Isso otimiza o treinamento, pois reduz drasticamente a necessidade de técnicas de *Undersampling* ou *Oversampling* agressivas, preservando a variância original dos dados.
* **Valor de Negócio:** Este critério aproxima o modelo de cenários reais da indústria de streaming (como Netflix e Crunchyroll), onde a métrica de sucesso não é apenas o "play", mas a retenção com avaliação positiva.

---

## 4. Filtros de Integridade e Qualidade
**Onde foi utilizado:** Cláusula `WHERE` da `master_table`.

**O que foi feito:**
* Filtro `r.score > 0` e `r.anime_id IS NOT NULL`.

**Por que:**
* **Remoção de Ruído:** Notas "0" indicam ausência de avaliação. Removê-las evita distorcer a média de preferência dos usuários.

---

## Resultados Finais Obtidos
* **Volume de Dados:** **~27.7 milhões** de registros validados.
* **Qualidade:** Mínimo de 1 episódio e máximo coerente (ex: Doraemon 1787).

In [ ]:
import matplotlib.pyplot as plt

pdf = spark.sql("SELECT episodes_raw, episodes_treated FROM comparison_table").toPandas()

# 1. Histogramas com Escala Logarítmica para melhor leitura da cauda
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Dados Crus - Usando escala log no eixo Y para ver as barras pequenas
ax1.hist(pdf['episodes_raw'], bins=100, color='salmon', edgecolor='black', alpha=0.7, log=True)
ax1.set_title('Distribuição Dados Crus (Escala Log no Y)')
ax1.set_xlabel('Nº de Episódios')
ax1.set_ylabel('Frequência (Log)')
ax1.grid(True, which="both", ls="-", alpha=0.2)

# Dados Tratados - Zoom no intervalo real (0 a 500)
ax2.hist(pdf['episodes_treated'], bins=50, color='seagreen', edgecolor='black', alpha=0.7)
ax2.set_title('Distribuição Dados Tratados (Escala Linear)')
ax2.set_xlabel('Nº de Episódios')
ax2.set_xlim(0, 500)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# 2. Boxplot com "Zoom" na Mediana (Limitando o eixo X)
plt.figure(figsize=(10, 6))
plt.boxplot([pdf['episodes_raw'], pdf['episodes_treated']],
            vert=False,
            patch_artist=True,
            labels=['Dados Crus', 'Dados Tratados'],
            showfliers=False, # Remove as bolinhas brancas para focar na caixa
            boxprops=dict(facecolor='lightblue'))

plt.title("Foco na Mediana e Quartis (Outliers ocultos para legibilidade)")
plt.xlabel("Quantidade de Episódios")
# Limitamos o X para ver onde a "ação" acontece
plt.xlim(-10, 1000)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()


### 1. Distribuição de Dados Crus: Identificação de Anomalias
Este gráfico apresenta a frequência de episódios antes de qualquer intervenção de limpeza, utilizando uma **escala logarítmica** no eixo vertical (Y).

* **Objetivo:** Visualizar a "cauda longa" da distribuição, onde residem os erros sistêmicos.
* **Análise Técnica:** Devido à presença de valores extremos (até 17.500 episódios), uma escala linear tornaria as barras de erro invisíveis. A escala logarítmica permite confirmar que o dataset estava severamente inflado por um fator de 10x e por erros de preenchimento manual.

### 2. Distribuição de Dados Tratados: Normalização da Escala
O segundo histograma foca na massa de dados após a aplicação das regras de tratamento e correção do fator de inflação.

* **Objetivo:** Validar se a maioria dos animes agora reside em um intervalo plausível (0 a 500 episódios).
* **Análise Técnica:** Observa-se uma curva muito mais condizente com a realidade da indústria audiovisual. A limpeza removeu o ruído estatístico, permitindo que análises futuras (como a taxa de conclusão por fonte) não sejam distorcidas por valores impossíveis.

### 3. Boxplot de Amplitude: Estabilidade e Variabilidade
O diagrama de caixa (Boxplot) com foco na mediana é a prova definitiva da eficácia do saneamento dos dados. Ao ocultar visualmente os outliers extremos, conseguimos comparar a estrutura interna das amostras.

* **Mediana e Quartis:** Nos **Dados Crus**, a "caixa" (Intervalo Interquartil) é larga e deslocada para a direita, indicando alta incerteza e valores inflados. Nos **Dados Tratados**, a caixa é estreita e posicionada próxima à origem, demonstrando alta consistência.
* **Conclusão:** O "encolhimento" da caixa prova que o processo de limpeza padronizou os registros. O tratamento reduziu a variância e eliminou a volatilidade, garantindo que métricas de negócio sejam baseadas em dados íntegros e confiáveis.

---

### Análise Exploratória de Dados (EDA) e Insights

#### Insight 1: Verificando variedade de usuários

In [ ]:
#Insight 1: Verificando variedade de usuários
print("---Quantidade de Usuários distintos que compõem a table de ratings---")
dinstinct_users = spark.sql("""SELECT COUNT(DISTINCT username) as distinct_users FROM master_table""")
display(dinstinct_users)


#### Insight 2: Verificação de desbalanceamento

In [ ]:
print("--- Distribuição da antiga Classe Alvo ---")
class_weight = spark.sql("""
          SELECT
              CASE WHEN completed_only = 1 THEN 'Completou' ELSE 'Outros' END as status_final,
              COUNT(*) as total,
              ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM master_table), 2) as pct
          FROM master_table
          GROUP BY completed_only
          """)
display(class_weight)


#### Observa-se um grande desbalanceamento entre as classes!
A antiga classe alvo que apenas visava saber se o usuario completou ou não um anime apresentava uma discrepância exorbitante. Pensando nisso foi escolhido um novo target que inclui uma nova condição pensando no engajamento que o anime proporcionou.


In [ ]:
print("--- Verificação da Nova Classe Alvo (Engajamento: Completou + Nota >= 8) ---")

new_class_weight = spark.sql("""
          SELECT
              target,
              CASE
                  WHEN target = 1 THEN 'Engajado (Completo + Nota 8-10)'
                  ELSE 'Não Engajado (Outros ou Nota < 8)'
                  END as status_descricao,
              COUNT(*) as total,
              ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM master_table), 2) as pct
          FROM master_table
          GROUP BY target
          ORDER BY target DESC
          """)

display(new_class_weight)

# 2. Verificação Extra: Média de notas por Target
print("--- Validação Técnica: Média de Score por Categoria ---")
score_comparison = spark.sql("""
          SELECT
              target,
              ROUND(AVG(user_score), 2) as media_nota_usuario,
              MIN(user_score) as nota_minima,
              MAX(user_score) as nota_maxima
          FROM master_table
          GROUP BY target
          """)

display(score_comparison)


### O novo target demonstrou um resultado extremamente satisfatório beirando o 50/50!

#### Insight 3: Verificação de taxa de conclusão com base no source do anime


In [ ]:
print("--- Top 10 Fontes com maior Taxa de Conclusão ---")
conclusion_rate = spark.sql("""
          SELECT
              anime_source,
              COUNT(*) as total_avaliacoes,
              ROUND(AVG(target) * 100, 2) as enjoyment
          FROM master_table
          GROUP BY anime_source
          HAVING total_avaliacoes > 1000 -- Filtro para evitar fontes irrelevantes
          ORDER BY enjoyment DESC
              LIMIT 10
          """)
display(conclusion_rate)

#### Insight 4: O "Efeito Longa Duração" (Episódios vs. Engajamento)

In [ ]:
print("--- Taxa de Conclusão por Faixa de Episódios ---")
enjoyment_by_episodes = spark.sql("""
          SELECT
              CASE
                  WHEN CAST(total_episodes AS INT) <= 12 THEN '01. Curto (1-12)'
                  WHEN CAST(total_episodes AS INT) <= 26 THEN '02. Padrão (13-26)'
                  WHEN CAST(total_episodes AS INT) <= 100 THEN '03. Longo (27-100)'
                  WHEN CAST(total_episodes AS INT) > 100 THEN '04. Épico (100+)'
                  ELSE '05. Desconhecido'
                  END as faixa_tamanho,
              COUNT(*) as total_avaliacoes,
              ROUND(AVG(target) * 100, 2) as enjoyment,
              ROUND(AVG(CAST(user_score AS DOUBLE)), 2) as media_nota_usuario
          FROM master_table
          GROUP BY 1
          ORDER BY 1 ASC
          """)
display(enjoyment_by_episodes)

### Será realizado um leve undersample apenas a fins de comparativo de performance entre os modelos a serem treinados!

In [5]:
# 1. SQL Puro para calcular os volumes e a fração de ajuste
# Usamos subqueries para pegar os totais de cada classe
stats = spark.sql("""
                  SELECT
                      count_target_1,
                      count_target_0,
                      CAST(count_target_1 AS DOUBLE) / CAST(count_target_0 AS DOUBLE) as fraction
                  FROM (
                           SELECT
                                   (SELECT COUNT(*) FROM master_table WHERE target = 1) as count_target_1,
                                   (SELECT COUNT(*) FROM master_table WHERE target = 0) as count_target_0
                       )
                  """).collect()[0]

fraction = stats['fraction']
total_completou = stats['count_target_1']
total_outros = stats['count_target_0']

# 2. SQL Puro para separar as classes
df_completou = spark.sql("SELECT * FROM master_table WHERE target = 1")
df_outros = spark.sql("SELECT * FROM master_table WHERE target = 0")

# 3. Executando o Undersampling na classe majoritária (Target 0)
# O método sample ainda é usado no objeto DataFrame, pois o SQL padrão não tem
# uma função de sampling aleatório determinístico cross-plataforma performática
df_outros_sampled = df_outros.sample(withReplacement=False, fraction=fraction, seed=42)

# 4. União das tabelas via SQL Puro (UNION ALL)
df_outros_sampled.createOrReplaceTempView("v_outros_sampled")
df_completou.createOrReplaceTempView("v_completou")

balanced_df = spark.sql("""
                        SELECT * FROM v_completou
                        UNION ALL
                        SELECT * FROM v_outros_sampled
                        """)

# 5. Registro Final
balanced_df.createOrReplaceTempView("balanced_master_table")

percentage = (1 - fraction)*100

print(f"--- Undersampling SQL Strict Concluído ---")
print(f"Fração aplicada em 'Outros': {percentage:.2f}%")

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `master_table` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 8 pos 57;
'Project ['count_target_1, 'count_target_0, (cast('count_target_1 as double) / cast('count_target_0 as double)) AS fraction#26]
+- 'SubqueryAlias __auto_generated_subquery_name
   +- 'Project [scalar-subquery#22 [] AS count_target_1#23, scalar-subquery#24 [] AS count_target_0#25]
      :  :- 'Aggregate [unresolvedalias(count(1), None)]
      :  :  +- 'Filter ('target = 1)
      :  :     +- 'UnresolvedRelation [master_table], [], false
      :  +- 'Aggregate [unresolvedalias(count(1), None)]
      :     +- 'Filter ('target = 0)
      :        +- 'UnresolvedRelation [master_table], [], false
      +- O

### Balanceamento de Carga via Undersampling Aleatório

**Onde foi utilizado:** Processamento da `master_table` para geração da `balanced_master_table`.

**O que foi feito:**
Implementou-se uma rotina de reequilíbrio estatístico utilizando injeção de SQL ANSI e amostragem aleatória determinística:
1. **Cálculo de Proporção:** Utilizou-se subqueries escalares para extrair o volume de ambas as classes e calcular a fração necessária para equalização.
2. **Segmentação:** As classes foram isoladas via cláusula `WHERE`.
3. **Amostragem:** Aplicou-se o método de *sampling* sem reposição na classe majoritária (`target=0`), reduzindo-a proporcionalmente à classe minoritária.
4. **Reconstituição:** Os conjuntos foram unificados através do operador `UNION ALL`.

**Por que:**
* **Baseline 50/50:** O balanceamento absoluto simplifica a interpretação das métricas de performance (Acurácia, Precision, Recall). Em um dataset equilibrado, a acurácia base é 50%, tornando qualquer ganho do modelo facilmente mensurável.
* **Mitigação de Viés de Predição:** Garante que o algoritmo não desenvolva uma tendência de classificar novos usuários como "Não Engajados" apenas por probabilidade estatística da base original.
* **Rigor Estatístico:** O uso de `seed=42` garante a reprodutibilidade dos resultados, permitindo que outros pesquisadores ou auditores cheguem ao mesmo dataset balanceado.

In [4]:
# Salvando o dataset tratado
print("Salvando o dataset tratado e balanceado...")
df_master_table_cleaned_balanced = df_master.write \
    .mode("overwrite") \
    .parquet(f"{output_path}/master_table_undersample")
print("Dataset salvo com sucesso!")

Salvando o dataset tratado e balanceado...


NameError: name 'df_master' is not defined

In [25]:
df_balanced = spark.read.parquet(f"{output_path}/master_table_undersample")
df_balanced.createOrReplaceTempView("undersample_master")

df_model = df_balanced.select(
    "anime_score",
    "total_episodes",
    "anime_type",
    "anime_source",
    "target"
).dropna()





## Refazendo os Insights após balanceamento

#### Insight 1: Verificando variedade de usuários

In [ ]:
print("--- Quantidade de Usuários distintos (Undersample) ---")
balanced_distinct_users = spark.sql("SELECT COUNT(DISTINCT username) as distinct_users FROM undersample_master")
display(balanced_distinct_users)

#### Insight 2: Verificação de desbalanceamento


In [ ]:
balanced_classes = spark.sql("""
          SELECT
              target,
              COUNT(*) as total,
              ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM balanced_master_table), 2) as pct
          FROM balanced_master_table
          GROUP BY target
          """)
display(balanced_classes)

#### Insight 3: Verificação de taxa de conclusão com base no source do anime


In [ ]:
print("--- Top 10 Fontes com maior Taxa de Conclusão (Undersample) ---")
balanced_conclusion_rate = spark.sql("""
          SELECT
              anime_source,
              COUNT(*) as total_avaliacoes,
              ROUND(AVG(target) * 100, 2) as enjoyment
          FROM undersample_master
          GROUP BY anime_source
          HAVING total_avaliacoes > 500 -- Ajustado para o novo volume
          ORDER BY enjoyment DESC
              LIMIT 10
          """)
display(balanced_conclusion_rate)

#### Insight 4: O "Efeito Longa Duração" (Episódios vs. Engajamento)


In [ ]:
print("--- Taxa de Conclusão por Faixa de Episódios (Undersample) ---")
enjoyment_by_episodes_balanced = spark.sql("""
          SELECT
              CASE
                  WHEN CAST(total_episodes AS INT) <= 12 THEN '01. Curto (1-12)'
                  WHEN CAST(total_episodes AS INT) <= 26 THEN '02. Padrão (13-26)'
                  WHEN CAST(total_episodes AS INT) <= 100 THEN '03. Longo (27-100)'
                  WHEN CAST(total_episodes AS INT) > 100 THEN '04. Épico (100+)'
                  ELSE '05. Desconhecido'
                  END as faixa_tamanho,
              COUNT(*) as total_avaliacoes,
              ROUND(AVG(target) * 100, 2) as enjoyment,
              ROUND(AVG(CAST(user_score AS DOUBLE)), 2) as media_nota_usuario
          FROM undersample_master
          GROUP BY 1
          ORDER BY 1 ASC
          """)
display(enjoyment_by_episodes_balanced)


---

In [ ]:
import matplotlib.pyplot as plt

# 1. Coletando os dados consolidados via SQL (Agregação no Spark)
# O .collect() traz apenas as linhas resultantes (neste caso, 2 linhas)
dist_original = spark.sql("""
                          SELECT target, ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM master_table), 2) as pct
                          FROM master_table GROUP BY target ORDER BY target
                          """).collect()

dist_balanced = spark.sql("""
                          SELECT target, ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM balanced_master_table), 2) as pct
                          FROM balanced_master_table GROUP BY target ORDER BY target
                          """).collect()

# Preparando listas para o Matplotlib
labels = ['Outros (0)', 'Completou (1)']
valores_orig = [row['pct'] for row in dist_original]
valores_bal = [row['pct'] for row in dist_balanced]

# Criando a visualização
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico 1: Original
ax1.bar(labels, valores_orig, color=['salmon', 'seagreen'])
ax1.set_title('Distribuição Original (Desbalanceada)')
ax1.set_ylabel('Porcentagem (%)')
for i, v in enumerate(valores_orig):
    # Convertendo v para float para permitir a soma
    ax1.text(i, float(v) + 0.5, f"{v}%", ha='center', fontweight='bold')

# Gráfico 2: Balanceado
ax2.bar(labels, valores_bal, color=['salmon', 'seagreen'])
ax2.set_title('Distribuição Final (Balanceada)')
ax2.set_ylabel('Porcentagem (%)')
for i, v in enumerate(valores_bal):
    # Convertendo v para float para permitir a soma
    ax2.text(i, float(v) + 0.5, f"{v}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Agregação do Histograma via SQL para o Dataset Original
# Criamos 10 buckets (faixas) para as notas de 1 a 10
hist_orig_df = spark.sql("""
                         SELECT
                             floor(user_score) as bin,
                             COUNT(*) as count
                         FROM master_table
                         GROUP BY bin
                         ORDER BY bin
                         """).collect()

# 2. Agregação do Histograma via SQL para o Dataset Balanceado
hist_bal_df = spark.sql("""
                        SELECT
                            floor(user_score) as bin,
                            COUNT(*) as count
                        FROM balanced_master_table
                        GROUP BY bin
                        ORDER BY bin
                        """).collect()

# 3. Preparando os dados para o Matplotlib
# Extraímos os valores das linhas retornadas pelo collect
bins_orig = [float(row['bin']) for row in hist_orig_df]
counts_orig = [row['count'] for row in hist_orig_df]

bins_bal = [float(row['bin']) for row in hist_bal_df]
counts_bal = [row['count'] for row in hist_bal_df]

# Criando o gráfico
fig, ax = plt.subplots(figsize=(12, 6))

# Largura das barras
width = 0.4

# Plotando as barras lado a lado para comparação clara
ax.bar([b - width/2 for b in bins_orig], counts_orig, width=width, label='Original (Target Híbrido)', color='blue', alpha=0.7)
ax.bar([b + width/2 for b in bins_bal], counts_bal, width=width, label='Balanceado (50/50)', color='orange', alpha=0.7)

ax.set_title('Distribuição de Notas (User Score): Comparativo entre Datasets')
ax.set_xlabel('Nota (Bin)')
ax.set_ylabel('Frequência (Quantidade de Registros)')
ax.set_xticks(range(1, 11))
ax.legend()

plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Análise de Insights: O Impacto da Duração no Enjoyment

A comparação entre os datasets **Original** (27M de linhas) e **Balanced/Undersample** (26M de linhas) revela uma consistência impressionante nas métricas de comportamento do usuário, validando a eficácia da amostragem.

#### 1. A Consistência do Enjoyment (Taxa de Satisfação)
Observa-se que a taxa de **Enjoyment** (Completou + Nota >= 8) é significativamente menor em animes **Curtos (44.6%)** em comparação aos animes **Padrão e Longos (~55%)**.

* **Insight:** Embora animes curtos tenham o maior volume de avaliações (18M no original), eles possuem a menor taxa de satisfação real. Isso sugere que a facilidade de terminar um anime curto não garante que o usuário irá amá-lo.

#### 2. O Ponto Ideal: 13 a 100 Episódios
As faixas "Padrão" e "Longo" apresentam o maior equilíbrio entre retenção e qualidade percebida:
* **Taxa de Enjoyment:** Estabiliza em torno de **55.4%**.
* **Média de Nota:** Sobe para a casa dos **7.6 a 7.7**, superando os animes curtos (7.16).

#### 3. O Paradoxo do Grupo Épico (100+)
Os animes com mais de 100 episódios mantêm uma taxa de Enjoyment robusta (**51.84%**) e notas altas.
* **Fidelização:** Isso indica que, se um usuário ultrapassa a barreira inicial de uma obra monumental, a probabilidade de ele atingir um estado de "Alta Satisfação" é maior do que em obras casuais de 12 episódios.

#### 4. Validação do Undersampling
A comparação entre as imagens demonstra que o processo de Undersampling foi cirúrgico:
* As médias de nota (`media_nota_usuario`) permaneceram idênticas até a segunda casa decimal.
* As taxas de `enjoyment` flutuaram menos de 0.1%.

**Conclusão Estratégica:**
O modelo de Machine Learning não será enviesado pelo volume massivo de animes curtos. Graças ao balanceamento, o algoritmo terá sensibilidade para identificar o "usuário de nicho" que prefere obras densas e longas, onde a satisfação final tende a ser estatisticamente superior.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Comparação da Enjoyment Rate com base em faixas
faixas = ['Curto', 'Padrão', 'Longo', 'Épico']
taxa_orig = [44.65, 55.42, 55.55, 52.04]
taxa_bal = [44.67, 55.4, 55.59, 51.84]

plt.figure(figsize=(10, 6))
plt.plot(faixas, taxa_orig, marker='o', label='Taxa Original (Viesada)', color='blue', linestyle='--')
plt.plot(faixas, taxa_bal, marker='s', label='Taxa Real (Balanceada)', color='red', linewidth=2)

plt.title('Impacto do Balanceamento na Taxa de Conclusão')
plt.xlabel('Tamanho do Anime')
plt.ylabel('Completion Rate (%)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Modelo Preditivo — Decision Tree Classification

## Objetivo da Modelagem

O objetivo desta etapa foi desenvolver um modelo de classificação utilizando o algoritmo Decision Tree para prever se um usuário apresentaria alto nível de satisfação com determinado anime.

O target foi definido utilizando uma estratégia híbrida:

target = \begin{cases}1, & \text{se } user_score \geq 8 \text{ e } completed_only = 1 \ 0, & \text{caso contrário}\end{cases}

Dessa forma, o modelo não considera apenas notas altas, mas também o engajamento do usuário ao concluir a obra.

---

# Estratégia de Modelagem

Para evitar problemas de data leakage, foram removidos atributos diretamente relacionados ao target, como:

* `user_score`
* `completed_only`

Isso garante que o modelo aprenda padrões reais relacionados às características dos animes, e não apenas reproduza informações já contidas na variável alvo.

As features utilizadas foram:

* `anime_score`
* `total_episodes`
* `anime_type`
* `anime_source`

---

# Preparação dos Dados

Inicialmente, foi realizado o carregamento do dataset balanceado e a seleção das colunas utilizadas pelo modelo.

```python
df_model = df_balanced.select(
    "anime_score",
    "total_episodes",
    "anime_type",
    "anime_source",
    "target"
).dropna()
```

---

# Transformação de Variáveis Categóricas

As colunas categóricas foram transformadas utilizando `StringIndexer`, permitindo que o algoritmo pudesse processar atributos textuais numericamente.

```python
from pyspark.ml.feature import StringIndexer

type_indexer = StringIndexer(
    inputCol="anime_type",
    outputCol="anime_type_index"
)

source_indexer = StringIndexer(
    inputCol="anime_source",
    outputCol="anime_source_index"
)
```

---

# Criação do Vetor de Features

Após a indexação, foi utilizado o `VectorAssembler` para consolidar todas as variáveis em um único vetor de features.

```python
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "anime_score",
        "total_episodes",
        "anime_type_index",
        "anime_source_index"
    ],
    outputCol="features"
)
```

---

# Divisão Treino/Teste

O dataset foi dividido em:

* 80% para treinamento
* 20% para teste

```python
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)
```

---

# Construção do Modelo Decision Tree

O algoritmo utilizado foi o `DecisionTreeClassifier` da biblioteca MLlib do Apache Spark.

Parâmetros principais:

* `maxDepth = 5`
* `seed = 42`

```python
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    labelCol="target",
    featuresCol="features",
    maxDepth=5,
    seed=42
)
```

---

# Construção do Pipeline

Foi criado um pipeline contendo:

1. Indexação das variáveis categóricas
2. Criação do vetor de features
3. Treinamento da árvore de decisão

```python
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[
    type_indexer,
    source_indexer,
    assembler,
    dt
])
```

---

# Treinamento do Modelo

O treinamento foi realizado utilizando o conjunto de treino.

```python
model = pipeline.fit(train_df)
```

---

# Realização das Predições

Após o treinamento, o modelo foi aplicado sobre o conjunto de teste.

```python
predictions = model.transform(test_df)
```

---

# Avaliação do Modelo

Foram utilizadas as métricas Accuracy e F1-Score.

```python
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1_score = f1_evaluator.evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score: {f1_score:.4f}")
```

---

# Resultados Obtidos

## Métricas de Avaliação

| Métrica  | Resultado |
| -------- | --------- |
| Accuracy | 68.30%    |
| F1-Score | 68.26%    |

Os resultados demonstram que o modelo conseguiu identificar padrões relevantes utilizando apenas características estruturais dos animes.

O fato do F1-Score permanecer extremamente próximo da Accuracy indica que o modelo apresentou comportamento relativamente equilibrado entre as classes, sem forte tendência para apenas um dos grupos.

---

# Distribuição das Classes após Balanceamento

O processo de undersampling resultou em uma distribuição significativamente mais equilibrada entre as classes:

| Classe     | Quantidade |
| ---------- | ---------- |
| Target = 1 | 13.346.375 |
| Target = 0 | 14.382.948 |

Código utilizado:

```python
df_balanced.groupBy("target").count().show()
```

Essa etapa foi fundamental para reduzir o viés do modelo em relação à classe majoritária e melhorar a capacidade de generalização.

---

# Matriz de Confusão

Código utilizado:

```python
predictions.groupBy("target", "prediction").count().show()
```

Resultado:

| Target Real | Predição | Quantidade |
| ----------- | -------- | ---------- |
| 1           | 0        | 938.917    |
| 0           | 0        | 2.038.946  |
| 1           | 1        | 1.728.154  |
| 0           | 1        | 809.434    |

## Interpretação

O modelo apresentou bom desempenho na identificação de usuários com alta satisfação (`target = 1`), alcançando mais de 1.7 milhão de previsões corretas positivas.

Além disso, o modelo também manteve boa capacidade de reconhecer casos negativos, demonstrando equilíbrio entre sensibilidade e especificidade.

---

# Importância das Features

Código utilizado:

```python
tree_model = model.stages[-1]

print(tree_model.featureImportances)
```

Resultado:

```python
(4,[0,1],[0.9996387583406208,0.00036124165937925334])
```

## Interpretação

O resultado demonstra que o atributo `anime_score` foi, de forma extremamente dominante, o fator mais relevante para as decisões da árvore.

Isso indica que a nota média global do anime possui forte correlação com a probabilidade de satisfação dos usuários.

Já o atributo `total_episodes` apresentou influência mínima, sugerindo que a duração da obra possui impacto secundário quando comparada à percepção geral de qualidade.

---

# Exemplos de Predição

Código utilizado:

```python
predictions.select(
    "anime_score",
    "anime_type",
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)
```

Exemplo observado:

| anime_score | anime_type | target | prediction |
| ----------- | ---------- | ------ | ---------- |
| 1.89        | OVA        | 0      | 0          |

Esse comportamento demonstra que o modelo conseguiu aprender relações coerentes entre qualidade percebida do anime e satisfação final dos usuários.

---

# Conclusão

O modelo Decision Tree apresentou desempenho consistente mesmo após a remoção de atributos diretamente relacionados ao target, evitando vazamento de dados e garantindo maior legitimidade estatística ao experimento.

Os resultados obtidos demonstram que características estruturais dos animes possuem capacidade relevante de prever padrões de satisfação dos usuários, especialmente através da variável `anime_score`.

Além disso, o uso do Apache Spark permitiu o treinamento do modelo sobre dezenas de milhões de registros, reforçando a aplicação prática de técnicas de Big Data e Machine Learning em larga escala.
